# Unidad 1: Profundización en Programación Estructurada y Algoritmos
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

En el ámbito de los **Negocios Digitales**, el desarrollo de software no se limita a escribir scripts aislados para analizar datos. Diseñar productos digitales escalables, integrar pasarelas de pago o estructurar pipelines de automatización exige un dominio profundo de la lógica estructurada, el control de flujo y la modularización en Python.

Un código profesional debe ser legible, modular, tolerante a fallos y alineado con los estándares de la industria (PEP 8). En esta unidad consolidaremos las bases algorítmicas, avanzaremos en el diseño de funciones y aprenderemos a gestionar excepciones de forma estructurada para prevenir caídas de producción en nuestras aplicaciones.

### Objetivos de Aprendizaje:
1. Consolidar el uso de sintaxis fundamental de Python (loops, condicionales y colecciones).
2. Manejar tipos de datos avanzados y estructuras de datos dinámicas.
3. Controlar errores mediante bloques `try-except-else-finally` y diseñar excepciones personalizadas.
4. Crear funciones modulares, reutilizables y tipadas con anotación de tipos (type hinting).
5. Aplicar buenas prácticas de código siguiendo los lineamientos de PEP 8.


## 1. Repaso y Consolidación de Sintaxis Fundamental

Antes de pasar a arquitecturas más complejas, repasemos herramientas fundamentales de control de flujo y colecciones, enfocadas en la optimización del código (como list y dict comprehensions).


In [1]:
# Ejemplo de List Comprehension para procesar una lista de montos de transacciones en bruto
montos_sucios = [" $120.50 ", " $45.00", "$9.99 ", "  $1500.00  "]

# Limpiamos y convertimos a float en una sola línea
montos_limpios = [float(monto.strip().replace("$", "")) for monto in montos_sucios]
print("Montos limpios:", montos_limpios)

# Filtrar transacciones de alto valor (mayores a 50)
transacciones_vip = [monto for monto in montos_limpios if monto > 50]
print("Transacciones VIP (> 50):", transacciones_vip)


Montos limpios: [120.5, 45.0, 9.99, 1500.0]
Transacciones VIP (> 50): [120.5, 1500.0]


### Estructuras de Datos Avanzadas: Diccionarios y Colecciones

Los diccionarios son el estándar de facto para representar payloads de APIs y configuraciones. Python cuenta con el módulo `collections` que provee estructuras de datos muy útiles para el día a día en un negocio digital (como `defaultdict` y `Counter`).


In [2]:
from collections import defaultdict, Counter

# Imaginemos un log de eventos de navegación en nuestro e-commerce
eventos_web = [
    ("user_1", "page_view"),
    ("user_2", "add_to_cart"),
    ("user_1", "add_to_cart"),
    ("user_3", "page_view"),
    ("user_1", "checkout_click"),
    ("user_2", "checkout_click")
]

# Agrupar eventos por usuario usando defaultdict
eventos_por_usuario = defaultdict(list)
for usuario, evento in eventos_web:
    eventos_por_usuario[usuario].append(evento)

print("Eventos por usuario:")
for usuario, eventos in eventos_por_usuario.items():
    print(f" - {usuario}: {eventos}")

# Contar frecuencias de eventos en todo el sitio
conteo_eventos = Counter([evento for _, evento in eventos_web])
print("\nFrecuencia total de eventos:", dict(conteo_eventos))


Eventos por usuario:
 - user_1: ['page_view', 'add_to_cart', 'checkout_click']
 - user_2: ['add_to_cart', 'checkout_click']
 - user_3: ['page_view']

Frecuencia total de eventos: {'page_view': 2, 'add_to_cart': 2, 'checkout_click': 2}


## 2. Gestión Estructurada de Excepciones

En producción, **los errores van a ocurrir**: la conexión de red con una pasarela de pago puede caerse, una base de datos puede estar saturada o un cliente puede mandar un campo vacío. Si no controlamos esto, nuestra app se interrumpirá.

### Bloque `try-except-else-finally`


In [3]:
def procesar_descuento(precio_base, descuento):
    try:
        # Intentamos calcular el precio con descuento
        precio_final = precio_base - (precio_base * (descuento / 100))
    except ZeroDivisionError:
        print("Error: El descuento no puede dividirse por cero de esta forma.")
        precio_final = precio_base
    except TypeError as e:
        print(f"Error de tipos detectado: {e}")
        precio_final = None
    else:
        print("El descuento se calculó exitosamente.")
    finally:
        print("Operación de cálculo completada.")
    return precio_final

print("--- Caso Exitoso ---")
print("Total:", procesar_descuento(100.0, 15))

print("\n--- Caso Fallido (Tipo Incorrecto) ---")
print("Total:", procesar_descuento(100.0, "quince"))


--- Caso Exitoso ---
El descuento se calculó exitosamente.
Operación de cálculo completada.
Total: 85.0

--- Caso Fallido (Tipo Incorrecto) ---
Error de tipos detectado: unsupported operand type(s) for /: 'str' and 'int'
Operación de cálculo completada.
Total: None


### Excepciones Personalizadas para Modelos de Negocio

Para hacer que nuestro código sea más descriptivo, es una buena práctica heredar de la clase base `Exception` para crear errores de dominio propios de nuestro negocio.


In [4]:
# Definición de excepciones de negocio
class LimiteCreditoSuperadoError(Exception):
    def __init__(self, monto_compra, limite_disponible):
        self.monto_compra = monto_compra
        self.limite_disponible = limite_disponible
        super().__init__(f"No se pudo procesar la compra por ${monto_compra:.2f}. Límite disponible: ${limite_disponible:.2f}")

def checkout(monto, limite_usuario):
    if monto > limite_usuario:
        raise LimiteCreditoSuperadoError(monto, limite_usuario)
    return "Pago aprobado de forma exitosa!"

# Simulando el flujo de negocio
limite_tarjeta = 500.0
compras = [120.0, 450.0]

for compra in compras:
    try:
        print(f"Intentando compra por ${compra}...")
        resultado = checkout(compra, limite_tarjeta)
        print(resultado)
    except LimiteCreditoSuperadoError as e:
        print(f"ALERTA BACKEND: {e}")
        # Aquí enviaríamos una alerta o pediríamos otro método de pago


Intentando compra por $120.0...
Pago aprobado de forma exitosa!
Intentando compra por $450.0...
Pago aprobado de forma exitosa!


## 3. Creación de Funciones Modulares y Manejo de Módulos Locales

Escribir funciones modulares nos permite dividir problemas complejos en partes sencillas y reutilizar el código. En Python, es altamente recomendado utilizar **Type Hinting** (anotación de tipos) para mejorar la autocompletación y detectar errores antes de ejecutar el código.

### Modularización, Parámetros Dinámicos (`*args` y `**kwargs`) e Inmutabilidad


In [6]:
import json
from typing import Dict

# Función que acepta argumentos posicionales variables (*args) y palabras clave (**kwargs)
def registrar_pedido(cliente_id: int, *items: str, **detalles_envio: str) -> Dict:
    """
    Registra un pedido de forma modular con tipado estricto.
    """
    pedido = {
        "cliente_id": cliente_id,
        "productos": list(items),
        "envio": detalles_envio,
        "estado": "pendiente"
    }
    return pedido

# Creación de pedido con ítems variables y metadata de envío dinámica
pedido_ejemplo = registrar_pedido(
    1024,
    "Suscripción SaaS Pro", "Soporte Premium 24/7",
    direccion="Av. Corrientes 1500, CABA",
    metodo="Envío Express Digital",
    prioridad="Alta"
)

print("Pedido Registrado:")
print(json.dumps(pedido_ejemplo, indent=2, ensure_ascii=False))

Pedido Registrado:
{
  "cliente_id": 1024,
  "productos": [
    "Suscripción SaaS Pro",
    "Soporte Premium 24/7"
  ],
  "envio": {
    "direccion": "Av. Corrientes 1500, CABA",
    "metodo": "Envío Express Digital",
    "prioridad": "Alta"
  },
  "estado": "pendiente"
}


### Manejo de Módulos Locales

En proyectos reales de desarrollo, dividimos el código en diferentes archivos `.py` (módulos) y carpetas (paquetes).
En Google Colab o Jupyter, podemos simular la escritura de archivos locales mediante el comando mágico `%%writefile`.


In [7]:
%%writefile validador_negocio.py
# Este código se guardará en un archivo local llamado validador_negocio.py
import re

def es_email_valido(email: str) -> bool:
    """Valida si un string tiene formato de email corporativo o general."""
    patron = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'
    return bool(re.match(patron, email))

def es_cuit_valido(cuit: str) -> bool:
    """Valida formato básico de CUIT (11 dígitos sin guiones)."""
    return cuit.isdigit() and len(cuit) == 11


Writing validador_negocio.py


Ahora podemos importar las funciones de nuestro módulo recién creado tal como lo haríamos en un proyecto profesional local.


In [8]:
# Importamos el módulo local creado dinámicamente
from validador_negocio import es_email_valido, es_cuit_valido

emails = ["juan@empresa.com", "cliente_invalido.com"]
cuits = ["20351234567", "123-45"]

for em in emails:
    print(f"¿Email '{em}' es válido?:", es_email_valido(em))

for cu in cuits:
    print(f"¿CUIT '{cu}' es válido?:", es_cuit_valido(cu))


¿Email 'juan@empresa.com' es válido?: True
¿Email 'cliente_invalido.com' es válido?: False
¿CUIT '20351234567' es válido?: True
¿CUIT '123-45' es válido?: False


## 4. Buenas Prácticas de Código y Estándares (PEP 8)

El estándar **PEP 8** define las reglas de estilo de Python:
- Nombre de variables en minúscula separadas por guiones bajos (`mi_variable`).
- Nombre de clases en CamelCase (`ConfiguracionSaaS`).
- Sangrado (indentación) de 4 espacios (evitar tabuladores).
- Comentarios explicativos concisos y docstrings descriptivos para todas las funciones.

Herramientas automáticas recomendadas para verificar la calidad en proyectos locales:
1. **Black**: Formateador automático que reestructura tu código según los estándares más estrictos.
2. **Ruff / Flake8**: Linters que analizan estáticamente tu código en busca de bugs latentes, variables no usadas y violaciones de estilo.

---

## Desafío Práctico (Trabajo Práctico 1)

**Consigna de Negocio (Pipeline de Clientes):**
Tu startup de Negocios Digitales necesita procesar un lote de nuevos clientes (leads) en bruto. Debes implementar un pipeline modular.

1. Crea un módulo local llamado `procesador_leads.py` utilizando la celda mágica `%%writefile`.
2. Dentro del módulo, define una excepción personalizada llamada `LeadInvalidoError` que contenga el email del lead y la razón de la falla.
3. Escribe una función `limpiar_y_validar_lead(lead: dict) -> dict` que reciba un diccionario del lead (con campos `nombre`, `email`, y `cuit`) y devuelva un diccionario limpio (con espacios eliminados y campos validados).
   - Debe lanzar `LeadInvalidoError` si el email no es válido (usa la función del módulo `validador_negocio` creado antes) o si el cuit no es numérico de 11 dígitos.
4. Escribe una función principal en el cuaderno que itere sobre la lista de leads provista abajo, procese cada uno capturando excepciones de forma que un lead inválido **no interrumpa** el procesamiento de los demás, e imprima un reporte final de leads exitosos y leads fallidos.

A continuación, implementa tu solución y pruébala.


In [9]:
"""Módulo para la gestión y configuración de servicios SaaS.

Cumple con las directrices de estilo PEP 8:
- Nombres de clases en CapWords (CamelCase).
- Nombres de funciones y variables en snake_case.
- Indentación estándar de 4 espacios.
- Docstrings descriptivos (PEP 257) y tipado estricto (PEP 484).
"""

from typing import Dict, List, Optional


class ConfiguracionSaaS:
    """Representa la configuración base para una instancia de servicio SaaS.

    Attributes:
        nombre_servicio: Nombre del módulo o servicio SaaS.
        limite_usuarios: Cantidad máxima de usuarios admitidos.
        activo: Estado operativo de la configuración.
    """

    # Atributo de clase en snake_case
    version_esquema: str = "2.1.0"

    def __init__(
        self,
        nombre_servicio: str,
        limite_usuarios: int,
        activo: bool = True,
    ) -> None:
        """Inicializa una nueva instancia de configuración.

        Args:
            nombre_servicio: Identificador del servicio.
            limite_usuarios: Capacidad máxima de usuarios permitidos.
            activo: Define si la configuración entra en vigencia de inmediato.
        """
        self.nombre_servicio = nombre_servicio
        self.limite_usuarios = limite_usuarios
        self.activo = activo
        self._parametros_adicionales: Dict[str, str] = {}

    def agregar_parametro(self, clave: str, valor: str) -> None:
        """Agrega un parámetro de configuración personalizado.

        Args:
            clave: Nombre de la variable de entorno o propiedad.
            valor: Valor asociado al parámetro.
        """
        self._parametros_adicionales[clave] = valor

    def exportar_resumen(self) -> Dict[str, object]:
        """Genera un diccionario con el estado actual del servicio.

        Returns:
            Dict con los datos clave de configuración.
        """
        return {
            "servicio": self.nombre_servicio,
            "limite": self.limite_usuarios,
            "activo": self.activo,
            "parametros_extra": self._parametros_adicionales,
        }


def verificar_compatibilidad(
    configs: List[ConfiguracionSaaS],
    minimo_usuarios: int = 10,
) -> List[str]:
    """Filtra y devuelve los nombres de servicios aptos para alta demanda.

    Args:
        configs: Lista de configuraciones a evaluar.
        minimo_usuarios: Umbral mínimo requerido.

    Returns:
        Lista de nombres de los servicios que cumplen el criterio.
    """
    servicios_aptos: List[str] = []

    for item in configs:
        if item.activo and item.limite_usuarios >= minimo_usuarios:
            servicios_aptos.append(item.nombre_servicio)

    return servicios_aptos


# --- Demostración de uso ---
if __name__ == "__main__":
    servicio_auth = ConfiguracionSaaS(
        nombre_servicio="AutenticacionOAuth",
        limite_usuarios=500,
        activo=True,
    )
    servicio_auth.agregar_parametro("timeout_segundos", "30")

    servicio_reportes = ConfiguracionSaaS(
        nombre_servicio="GeneradorReportes",
        limite_usuarios=5,
        activo=False,
    )

    lista_servicios = [servicio_auth, servicio_reportes]
    aptos = verificar_compatibilidad(lista_servicios, minimo_usuarios=50)

    print("Resumen de servicio:", servicio_auth.exportar_resumen())
    print("Servicios aptos para alta demanda:", aptos)

Resumen de servicio: {'servicio': 'AutenticacionOAuth', 'limite': 500, 'activo': True, 'parametros_extra': {'timeout_segundos': '30'}}
Servicios aptos para alta demanda: ['AutenticacionOAuth']


In [13]:
import importlib
import re
import sys

# =====================================================================
# PASO 1: Creación del módulo validador_negocio.py
# =====================================================================
with open("validador_negocio.py", "w", encoding="utf-8") as f:
    f.write('''import re

def validar_email(email: str) -> bool:
    """Valida que el formato del correo electrónico sea correcto."""
    patron = r"^[\\w\\.-]+@[\\w\\.-]+\\.\\w+$"
    return bool(re.match(patron, email.strip()))
''')

# Si ya estaba cargado en caché de Python, se recarga; si no, se importa
if "validador_negocio" in sys.modules:
    import validador_negocio
    importlib.reload(validador_negocio)
else:
    import validador_negocio


# =====================================================================
# PASO 2: Creación del módulo procesador_leads.py
# =====================================================================
with open("procesador_leads.py", "w", encoding="utf-8") as f:
    f.write('''import re
from validador_negocio import validar_email


class LeadInvalidoError(Exception):
    """Excepción lanzada cuando un lead no cumple con las validaciones de negocio."""
    def __init__(self, email: str, razon: str):
        self.email = email
        self.razon = razon
        super().__init__(f"Lead inválido [{email}]: {razon}")


def limpiar_y_validar_lead(lead: dict) -> dict:
    """Limpia cadenas y valida los campos nombre, email y cuit de un lead."""
    nombre = str(lead.get("nombre", "")).strip()
    email = str(lead.get("email", "")).strip()
    cuit_limpio = re.sub(r"[^\\d]", "", str(lead.get("cuit", "")))

    if not validar_email(email):
        raise LeadInvalidoError(email, "Formato de correo electrónico inválido.")

    if len(cuit_limpio) != 11:
        raise LeadInvalidoError(
            email,
            f"El CUIT debe contener exactamente 11 dígitos numéricos (recibidos: {len(cuit_limpio)})."
        )

    return {
        "nombre": nombre,
        "email": email,
        "cuit": cuit_limpio
    }
''')

if "procesador_leads" in sys.modules:
    import procesador_leads
    importlib.reload(procesador_leads)


# =====================================================================
# PASO 3: Importación y Ejecución del Pipeline
# =====================================================================
from procesador_leads import LeadInvalidoError, limpiar_y_validar_lead

leads_en_bruto = [
    {"nombre": "  Ana Ruiz  ", "email": "ana.ruiz@techcorp.com", "cuit": "27-35123456-4"},
    {"nombre": "Carlos Gómez", "email": "carlos.gomez_sin_arroba.com", "cuit": "20-30123456-8"},
    {"nombre": "Lucía Méndez ", "email": "lucia@startup.io", "cuit": "27-28999888-0"},
    {"nombre": "Marcos Paz", "email": "marcos@empresa.com", "cuit": "20-1234-5"},
    {"nombre": "  Sofía Blanco", "email": "sofia.blanco@service.ar", "cuit": "23381112224"},
    {"nombre": "Javier Díaz", "email": "javier@invalido", "cuit": "20334445558"}
]

leads_exitosos = []
leads_fallidos = []

for lead in leads_en_bruto:
    try:
        lead_procesado = limpiar_y_validar_lead(lead)
        leads_exitosos.append(lead_procesado)
    except LeadInvalidoError as error:
        leads_fallidos.append({
            "email": error.email,
            "razon": error.razon,
            "datos_originales": lead
        })

print("=" * 70)
print("                   REPORTE DEL PIPELINE DE LEADS")
print("=" * 70)

print(f"\n✔ LEADS EXITOSOS ({len(leads_exitosos)}):")
for lead in leads_exitosos:
    print(f"  - {lead['nombre']:<15} | CUIT: {lead['cuit']} | Email: {lead['email']}")

print(f"\n✖ LEADS RECHAZADOS ({len(leads_fallidos)}):")
for fallo in leads_fallidos:
    print(f"  - Identificador: {fallo['email']:<28} | Causa: {fallo['razon']}")

print("\n" + "=" * 70)

                   REPORTE DEL PIPELINE DE LEADS

✔ LEADS EXITOSOS (3):
  - Ana Ruiz        | CUIT: 27351234564 | Email: ana.ruiz@techcorp.com
  - Lucía Méndez    | CUIT: 27289998880 | Email: lucia@startup.io
  - Sofía Blanco    | CUIT: 23381112224 | Email: sofia.blanco@service.ar

✖ LEADS RECHAZADOS (3):
  - Identificador: carlos.gomez_sin_arroba.com  | Causa: Formato de correo electrónico inválido.
  - Identificador: marcos@empresa.com           | Causa: El CUIT debe contener exactamente 11 dígitos numéricos (recibidos: 7).
  - Identificador: javier@invalido              | Causa: Formato de correo electrónico inválido.

